In [1]:
# ============================================
# California Coastal Conditions Pipeline
# Stage 2 of 4: Scrape and clean California lighthouse locations
# Author: Brittany Blessie
# Description: Scrapes the California lighthouse table from Wikipedia HTML and applies
#              labeled transformation steps to produce a clean dataset with decimal coordinates.
# ============================================

In [2]:
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_lighthouses_in_California"

# Some sites block default script requests, so send a browser-like User-Agent header
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

# Pass the fetched HTML text to read_html instead of the URL
tables = pd.read_html(response.text)
len(tables)

C:\Users\brittany.blessie\AppData\Local\Temp\ipykernel_15344\3253908622.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


3

In [3]:
# Check the shape of each scraped table to find the lighthouse one
for i in range(len(tables)):
    print(i, tables[i].shape)

0 (51, 9)
1 (1, 7)
2 (4, 2)


The page returned three tables. Table 0 has 51 rows and 9 columns, which is the lighthouse table. The other two are small side tables, so table 0 is the one to use.

In [4]:
# Pull the lighthouse table (index 0) out of the list into its own DataFrame
lighthouses = tables[0]
lighthouses.head(10)

,Name,Image,Location,Coordinates,Year first lit,Automated,Year deactivated,Current Lens,Focal Height
0,Alcatraz Island Light,NaN,San Francisco (Alcatraz Island),37°49′34.5″N 122°25′19.8″W﻿ / ﻿37.826250°N 122...,1854 (First) 1909 (Current),1963,Active,DCB-24,214 ft (65 m)
1,Anacapa Island Light,NaN,Anacapa Island,34°00′57″N 119°21′34″W﻿ / ﻿34.015827°N 119.359...,1912 (First) 1932 (Current),1967[1],Active,DCB-24,277 ft (84 m)
2,Ano Nuevo Light,NaN,Año Nuevo Island,37°06′30″N 122°20′16″W﻿ / ﻿37.1083°N 122.3378°...,1890 (First) 1914 (Last),Never,1948 (Cut down in 1976),NaN,Unknown
3,Ballast Point Light,NaN,San Diego (Point Loma),32°41′11.0″N 117°13′57.0″W﻿ / ﻿32.686389°N 117...,1890,Never,1960A (Demolished),NaN,Unknown
4,Battery Point Light,NaN,Crescent City,41°44′39″N 124°12′11″W﻿ / ﻿41.744094°N 124.203...,1856,1953,Active (Inactive: 1965–1982),375mm,77 ft (23 m)
5,Cape Mendocino Light,NaN,Shelter Cove,40°26′23.66″N 124°24′21.71″W﻿ / ﻿40.4399056°N ...,1868,1951,1971B (Replaced),NaN,422 ft (129 m)
6,Carquinez Strait Light,NaN,Vallejo,38°04′04″N 122°12′50″W﻿ / ﻿38.067816°N 122.213...,1910,Never,1951 (Replaced by beacon),NaN,56 ft (17 m)
7,East Brother Island Light,NaN,Richmond,37°57′48″N 122°26′01″W﻿ / ﻿37.963233°N 122.433...,1874,1969,Active,FA 251,61 ft (19 m)
8,Farallon Island Light,NaN,San Francisco (Farallon Islands),37°41′56″N 123°00′06″W﻿ / ﻿37.698966°N 123.001...,1856,1972,Active,VRB-25,358 ft (109 m)
9,Fort Point Light,NaN,San Francisco,37°48′38″N 122°28′38.4″W﻿ / ﻿37.81056°N 122.47...,1855 (First) 1864 (Current),Never,1934 (Replaced by bridge),NaN,110 ft (34 m)


## Step 1: Select Needed Columns
Keep only Name, Location, and Coordinates from the scraped table. Name and Location identify each lighthouse, and Coordinates is the join key to the CalCOFI data.

In [5]:
# Step #1: Keep only the columns needed from the lighthouse table.
# Name and Location identify each lighthouse; Coordinates is the join key to the CalCOFI data.
lh_clean = lighthouses[["Name", "Location", "Coordinates"]]
lh_clean.shape

(51, 3)

In [6]:
# Look at the full text of the Coordinates column (no truncation) for the first 10 rows
for i in range(10):
    print(i, "|", lh_clean["Coordinates"].iloc[i])

0 | 37°49′34.5″N 122°25′19.8″W﻿ / ﻿37.826250°N 122.422167°W
1 | 34°00′57″N 119°21′34″W﻿ / ﻿34.015827°N 119.359548°W
2 | 37°06′30″N 122°20′16″W﻿ / ﻿37.1083°N 122.3378°W (Island coordinates)
3 | 32°41′11.0″N 117°13′57.0″W﻿ / ﻿32.686389°N 117.232500°W
4 | 41°44′39″N 124°12′11″W﻿ / ﻿41.744094°N 124.203099°W
5 | 40°26′23.66″N 124°24′21.71″W﻿ / ﻿40.4399056°N 124.4060306°W
6 | 38°04′04″N 122°12′50″W﻿ / ﻿38.067816°N 122.213832°W
7 | 37°57′48″N 122°26′01″W﻿ / ﻿37.963233°N 122.433643°W
8 | 37°41′56″N 123°00′06″W﻿ / ﻿37.698966°N 123.001651°W
9 | 37°48′38″N 122°28′38.4″W﻿ / ﻿37.81056°N 122.477333°W


This left me with 51 rows and 3 columns: Name, Location, and Coordinates. The other six columns, including an empty Image column, were dropped.

## Step 2-3: Extract Coordinates and Fix Longitude Sign
Use a regex pattern to pull decimal latitude and longitude out of the messy coordinate text, then multiply longitude by -1 since west longitudes are negative in decimal form.

In [7]:
# Step #2: Extract decimal latitude and longitude from the messy Coordinates text.
# The decimal values always appear right before °N (latitude) and °W (longitude), so a regex pattern pulls just those numbers out and ignores the CSS junk and 
# DMS text.
lh_clean = lh_clean.copy()
lh_clean["Latitude"] = lh_clean["Coordinates"].str.extract(r"(\d+\.\d+)°N").astype(float)
lh_clean["Longitude"] = lh_clean["Coordinates"].str.extract(r"(\d+\.\d+)°W").astype(float)

# Step #3: Make longitude negative, since W (west) longitudes are negative in decimal form.
lh_clean["Longitude"] = lh_clean["Longitude"] * -1

lh_clean[["Name", "Latitude", "Longitude"]].head(10)

,Name,Latitude,Longitude
0,Alcatraz Island Light,37.826250,-122.422167
1,Anacapa Island Light,34.015827,-119.359548
2,Ano Nuevo Light,37.108300,-122.337800
3,Ballast Point Light,32.686389,-117.232500
4,Battery Point Light,41.744094,-124.203099
5,Cape Mendocino Light,40.439906,-124.406031
6,Carquinez Strait Light,38.067816,-122.213832
7,East Brother Island Light,37.963233,-122.433643
8,Farallon Island Light,37.698966,-123.001651
9,Fort Point Light,37.810560,-122.477333


The regex pulled clean decimal latitude and longitude for every row, including the Alcatraz row that had CSS code in front of its coordinates. Longitudes are now negative, which is correct for the California coast.

## Step 4: Check for Failed Extractions
Count missing values in Latitude and Longitude to confirm the regex worked across all 50 rows.

In [8]:
# Step #4: Check for rows where the coordinate extraction failed (missing Latitude or Longitude).
lh_clean[["Latitude", "Longitude"]].isnull().sum()

Latitude     1
Longitude    1
dtype: int64

In [9]:
# Show the row(s) where coordinate extraction failed
lh_clean[lh_clean["Latitude"].isnull()]

,Name,Location,Coordinates,Latitude,Longitude
30,Point Knox Light,San Francisco,—N/a,NaN,NaN


The check found one row missing coordinates. Point Knox Light had only a placeholder dash in its coordinate cell on Wikipedia, so the regex correctly found nothing to extract. This row is dropped in the next step.

## Step 5: Drop Rows With No Coordinates
Remove Point Knox Light, which had no coordinates in the source. A lighthouse with no location cannot be matched to the CalCOFI data.

In [10]:
# Step #5: Drop rows with no coordinates (Point Knox Light had no lat/lon in the source).
# A lighthouse with no location cannot be matched to the CalCOFI data, so it is removed.
lh_clean = lh_clean.dropna(subset=["Latitude", "Longitude"])
lh_clean.shape

(50, 5)

Dropping the row with no coordinates left 50 lighthouses. Only Point Knox was removed, which matches what the missing-value check found.

## Step 6: Drop the Original Coordinates Column
Remove the messy Coordinates column now that clean Latitude and Longitude have been extracted from it.

In [11]:
# Step #6: Drop the original Coordinates column now that clean Latitude and Longitude are extracted.
lh_clean = lh_clean.drop(columns=["Coordinates"])
lh_clean.head(10)

,Name,Location,Latitude,Longitude
0,Alcatraz Island Light,San Francisco (Alcatraz Island),37.826250,-122.422167
1,Anacapa Island Light,Anacapa Island,34.015827,-119.359548
2,Ano Nuevo Light,Año Nuevo Island,37.108300,-122.337800
3,Ballast Point Light,San Diego (Point Loma),32.686389,-117.232500
4,Battery Point Light,Crescent City,41.744094,-124.203099
5,Cape Mendocino Light,Shelter Cove,40.439906,-124.406031
6,Carquinez Strait Light,Vallejo,38.067816,-122.213832
7,East Brother Island Light,Richmond,37.963233,-122.433643
8,Farallon Island Light,San Francisco (Farallon Islands),37.698966,-123.001651
9,Fort Point Light,San Francisco,37.810560,-122.477333


Removing the original Coordinates column left four clean columns: Name, Location, Latitude, and Longitude. The messy source text is no longer in the dataset.

## Step 7: Reset the Index
Renumber the rows from zero after dropping a row, so the final dataset has a clean, continuous index.

In [12]:
# Step #7: Reset the index after dropping a row so the final dataset is numbered cleanly.
lh_clean = lh_clean.reset_index(drop=True)
lh_clean.head(10)

,Name,Location,Latitude,Longitude
0,Alcatraz Island Light,San Francisco (Alcatraz Island),37.826250,-122.422167
1,Anacapa Island Light,Anacapa Island,34.015827,-119.359548
2,Ano Nuevo Light,Año Nuevo Island,37.108300,-122.337800
3,Ballast Point Light,San Diego (Point Loma),32.686389,-117.232500
4,Battery Point Light,Crescent City,41.744094,-124.203099
5,Cape Mendocino Light,Shelter Cove,40.439906,-124.406031
6,Carquinez Strait Light,Vallejo,38.067816,-122.213832
7,East Brother Island Light,Richmond,37.963233,-122.433643
8,Farallon Island Light,San Francisco (Farallon Islands),37.698966,-123.001651
9,Fort Point Light,San Francisco,37.810560,-122.477333


In [13]:
# Final cleaned lighthouse dataset (all 50 rows)
lh_clean

,Name,Location,Latitude,Longitude
0,Alcatraz Island Light,San Francisco (Alcatraz Island),37.826250,-122.422167
1,Anacapa Island Light,Anacapa Island,34.015827,-119.359548
2,Ano Nuevo Light,Año Nuevo Island,37.108300,-122.337800
3,Ballast Point Light,San Diego (Point Loma),32.686389,-117.232500
4,Battery Point Light,Crescent City,41.744094,-124.203099
5,Cape Mendocino Light,Shelter Cove,40.439906,-124.406031
6,Carquinez Strait Light,Vallejo,38.067816,-122.213832
7,East Brother Island Light,Richmond,37.963233,-122.433643
8,Farallon Island Light,San Francisco (Farallon Islands),37.698966,-123.001651
9,Fort Point Light,San Francisco,37.810560,-122.477333


The index now runs cleanly from 0 to 49 after the dropped row. The final dataset has 50 California lighthouses, each with a name, location, and usable decimal coordinates.

## Data Notes and Limitations

The transformations I made to the lighthouse website data were as follows: I scraped the California lighthouse table from Wikipedia, kept only the name, location, and coordinates, extracted decimal latitude and longitude from the messy coordinate text, made longitude negative to reflect western hemisphere values, dropped one lighthouse that had no coordinates in the source, removed the original coordinate column, and reset the index. On legal and regulatory guidelines, Wikipedia content is published under a Creative Commons license that allows reuse with attribution, and the data contains no personal or sensitive information, so there are no privacy concerns. The main risk created by these transformations is that scraped web data can change or be inaccurate, since Wikipedia is openly editable, so the coordinates are only as reliable as the page at the time I pulled it. There is also a smaller risk that my regex extraction could have pulled a wrong value if a cell used an unusual format, though I checked for failed extractions and found only the one row with no coordinates at all. As for assumptions, I assumed the decimal coordinates listed on Wikipedia are accurate, and I assumed a lighthouse with no coordinates could be dropped since it cannot be matched to the ocean data by location. On sourcing and credibility, I pulled the data directly from the Wikipedia page and verified the extraction by printing the results and confirming the coordinates fell in the expected range for the California coast. The data was acquired ethically, since it is public information gathered through standard web scraping of an openly licensed page, using a normal request that did not overload the site. To mitigate these risks, I recorded the exact source URL and the date I scraped it, kept the raw scraped table before cleaning so the original is preserved, and would cross-check the coordinates against an authoritative source such as the Coast Guard light list before using them for anything critical.

In [14]:
# Save the cleaned lighthouse data so the weather pull can read the coordinates
lh_clean.to_csv("lighthouses_clean.csv", index=False)